# 02 — Signal Lab : Feature Engineering
Extraction des biomarqueurs physiologiques à partir des données 1Hz.

**Features calculées :**
- Efficiency Factor (global + spécifique à 3'40"/km)
- Cardiac Decoupling (dérive cardiaque)
- TRIMP (charge interne)
- Zones FC
- Normalized Power

In [ ]:
import sys
sys.path.insert(0, '..')

from src.layer_a_signal.ingest_fit import parse_fit, get_session_summary
from src.layer_a_signal.features import (
    clean_signal, session_features, load_params,
    cardiac_decoupling, get_running_mask
)
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

params = load_params('../config/params.yaml')

# Charger et nettoyer
df_raw, df_rr = parse_fit('../data/raw/longue.fit')
df = clean_signal(df_raw, params)

print(f"Points bruts : {len(df_raw)} | Points nettoyés : {len(df)}")
print(f"Durée : {(df.index[-1] - df.index[0])}")

Points bruts : 4977 | Points nettoyés : 4977
Durée : 0 days 01:23:25


## 1. Résumé complet de la séance

In [4]:
feats = session_features(df, params)

# Affichage propre
print("=" * 50)
print(f"  Distance      : {feats['distance_m']/1000:.2f} km")
print(f"  Durée course  : {feats['duration_running_s']//60}min {feats['duration_running_s']%60}s")
print(f"  FC moy / max  : {feats['hr_mean']:.0f} / {feats['hr_max']:.0f} bpm")
print(f"  Vitesse moy   : {feats['speed_mean_kmh']:.1f} km/h")
print(f"  Puissance moy : {feats['power_mean']:.0f} W")
print(f"  Cadence moy   : {feats['cadence_mean']:.0f} rpm")
print("=" * 50)
print(f"  EF global     : {feats['ef_global']:.4f}")
print(f"  EF spécifique : {feats['ef_specific']:.4f}" if not np.isnan(feats['ef_specific']) else "  EF spécifique : N/A (pas assez de temps à l'allure cible)")
print(f"  Decoupling    : {feats['cardiac_decoupling']:.1f}%")
print(f"  TRIMP         : {feats['trimp']:.0f}")
print(f"  Norm. Power   : {feats['normalized_power']:.0f} W")
print("=" * 50)

# Zones FC
print("\n  Zones FC :")
for zone, data in feats['hr_zones'].items():
    bar = '█' * int(data['pct'] / 2)
    print(f"    {zone:15} ({data['range_bpm']:>9}) : {data['pct']:5.1f}% {bar}")

  Distance      : 16.12 km
  Durée course  : 82min 42s
  FC moy / max  : 144 / 172 bpm
  Vitesse moy   : 11.7 km/h
  Puissance moy : 245 W
  Cadence moy   : 82 rpm
  EF global     : 0.0226
  EF spécifique : N/A (pas assez de temps à l'allure cible)
  Decoupling    : -1.1%
  TRIMP         : 90
  Norm. Power   : 245 W

  Zones FC :
    z1_recovery     (  102-123) :   1.0% 
    z2_endurance    (  123-143) :  43.2% █████████████████████
    z3_tempo        (  143-164) :  55.1% ███████████████████████████
    z4_threshold    (  164-184) :   0.1% 
    z5_vo2max       (  184-205) :   0.0% 


## 2. Visualisation multi-signaux

In [5]:
mask = get_running_mask(df, params)
df_run = df.loc[mask]

# Lissage pour la lisibilité (rolling 10s)
smooth = 10

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=('Fréquence Cardiaque', 'Vitesse (km/h)', 'Puissance (W)', 'Cadence (rpm)'),
    vertical_spacing=0.05
)

# FC
fig.add_trace(go.Scatter(
    x=df_run.index, y=df_run['hr'].rolling(smooth).mean(),
    name='FC', line=dict(color='red', width=1)
), row=1, col=1)

# Vitesse
fig.add_trace(go.Scatter(
    x=df_run.index, y=df_run['speed_kmh'].rolling(smooth).mean(),
    name='Vitesse', line=dict(color='blue', width=1)
), row=2, col=1)
# Ligne allure cible
target_kmh = params['athlete']['target_speed_ms'] * 3.6
fig.add_hline(y=target_kmh, line_dash='dash', line_color='green',
              annotation_text=f"Cible {target_kmh:.1f} km/h", row=2, col=1)

# Puissance
if 'power' in df_run.columns:
    fig.add_trace(go.Scatter(
        x=df_run.index, y=df_run['power'].rolling(smooth).mean(),
        name='Power', line=dict(color='purple', width=1)
    ), row=3, col=1)

# Cadence
if 'cadence' in df_run.columns:
    fig.add_trace(go.Scatter(
        x=df_run.index, y=df_run['cadence'].rolling(smooth).mean(),
        name='Cadence', line=dict(color='orange', width=1)
    ), row=4, col=1)

fig.update_layout(height=800, showlegend=False, template='plotly_white',
                  title='Sortie Longue — Signaux Bruts (lissés 10s)')
fig.show()

## 3. Cardiac Decoupling — Visualisation de la dérive

In [8]:
# Calcul du ratio EF glissant (fenêtre 60s)
df_run = df.loc[get_running_mask(df, params)].dropna(subset=['speed_ms', 'hr']).copy()

window_ef = 60  # 1 minute
df_run['ef_rolling'] = (
    df_run['speed_ms'].rolling(window_ef).mean() / 
    df_run['hr'].rolling(window_ef).mean()
)

# Temps écoulé en minutes pour l'axe X (évite les problèmes Plotly + tz-aware Timestamps)
t0 = df_run.index[0]
df_run['elapsed_min'] = [(t - t0).total_seconds() / 60 for t in df_run.index]
mid_min = df_run['elapsed_min'].iloc[len(df_run)//2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df_run['elapsed_min'], y=df_run['ef_rolling'],
    name='EF (rolling 60s)', line=dict(color='teal', width=1.5)
))

fig2.add_vline(x=mid_min, line_dash='dash', line_color='gray',
               annotation_text='Moitié')

# Trend linéaire
x_num = df_run['elapsed_min'].values
ef_vals = df_run['ef_rolling'].values
valid = ~np.isnan(ef_vals)
if valid.sum() > 10:
    z = np.polyfit(x_num[valid], ef_vals[valid], 1)
    trend = np.polyval(z, x_num)
    fig2.add_trace(go.Scatter(
        x=df_run['elapsed_min'], y=trend,
        name='Tendance', line=dict(color='red', dash='dot', width=2)
    ))

decoupling_val = cardiac_decoupling(df, params)
color = 'green' if abs(decoupling_val) < 5 else 'red'

fig2.update_layout(
    height=400, template='plotly_white',
    title=f'Cardiac Decoupling : <span style="color:{color}">{decoupling_val:.1f}%</span> {"✅" if abs(decoupling_val)<5 else "⚠️"}',
    xaxis_title='Temps (min)',
    yaxis_title='EF ratio (speed/hr)'
)
fig2.show()

## 4. Allure vs FC — Efficiency Factor dans le temps

In [9]:
# On découpe la séance en blocs de 5 minutes pour voir l'évolution de l'EF
df_run = df.loc[get_running_mask(df, params)].dropna(subset=['speed_ms', 'hr']).copy()

block_size = 300  # 5 minutes
n_blocks = len(df_run) // block_size

blocks = []
for i in range(n_blocks):
    chunk = df_run.iloc[i*block_size:(i+1)*block_size]
    blocks.append({
        'block': i + 1,
        'time_min': i * 5,
        'hr_mean': chunk['hr'].mean(),
        'speed_kmh': chunk['speed_kmh'].mean(),
        'ef': chunk['speed_ms'].mean() / chunk['hr'].mean(),
        'power_mean': chunk['power'].mean() if 'power' in chunk.columns else np.nan
    })

df_blocks = pd.DataFrame(blocks)

fig3 = make_subplots(specs=[[{'secondary_y': True}]])

fig3.add_trace(go.Bar(
    x=df_blocks['time_min'], y=df_blocks['ef'],
    name='EF', marker_color='teal', opacity=0.7
), secondary_y=False)

fig3.add_trace(go.Scatter(
    x=df_blocks['time_min'], y=df_blocks['hr_mean'],
    name='FC moy', line=dict(color='red', width=2)
), secondary_y=True)

fig3.update_layout(
    height=400, template='plotly_white',
    title='Évolution EF & FC par blocs de 5 min',
    xaxis_title='Temps (min)'
)
fig3.update_yaxes(title_text='EF (speed/hr)', secondary_y=False)
fig3.update_yaxes(title_text='FC (bpm)', secondary_y=True)
fig3.show()